In [ ]:
from google.colab import files

uploaded = files.upload()
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.move(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip -q install kaggle
!kaggle --version

In [ ]:
!kaggle datasets list -s plantvillage

In [ ]:
!kaggle datasets download \
    -d abdallahalidev/plantvillage-dataset \
    -p /content/plantvillage \
    --unzip
import os

dataset_path = "/content/plantvillage"

for root, dirs, files in os.walk(dataset_path):
    if files:
        print("Folder:", root)
        print("Number of files:", len(files))
        print()
        if root.count(os.sep) >= 4:
            dirs[:] = []

In [ ]:
import os

dataset_path = "/content/plantvillage/plantvillage dataset/color"

classes = [
    name for name in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, name))
]

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

total_images = 0

for class_name in sorted(classes):
    class_path = os.path.join(dataset_path, class_name)
    count = sum(
        1 for name in os.listdir(class_path)
        if name.lower().endswith(image_extensions)
    )
    total_images += count

print("Number of classes:", len(classes))
print("Total color images:", total_images)

import os
import hashlib
import pandas as pd
from collections import defaultdict

dataset_path = "/content/plantvillage/plantvillage dataset/color"
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

hash_groups = defaultdict(list)

for root, dirs, files in os.walk(dataset_path):
    for filename in files:
        if filename.lower().endswith(image_extensions):
            path = os.path.join(root, filename)

            md5 = hashlib.md5()

            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    md5.update(chunk)

            hash_groups[md5.hexdigest()].append(path)

rows = []

for md5_hash, paths in hash_groups.items():
    for path in paths:
        rows.append({
            "image_path": path,
            "label": os.path.basename(os.path.dirname(path)),
            "md5": md5_hash,
            "duplicate_count": len(paths),
            "is_duplicate": len(paths) > 1
        })

df = pd.DataFrame(rows)

df.to_csv("/content/plantvillage_md5_audit.csv", index=False)

print("Total images scanned:", len(df))
print("Unique MD5 hashes:", df["md5"].nunique())
print("Images belonging to duplicate groups:", df["is_duplicate"].sum())
print("Extra copies that can be removed:",
      len(df) - df["md5"].nunique())
print("Audit CSV saved.")

Number of classes: 38
Total color images: 54305
Total images scanned: 54305
Unique MD5 hashes: 54284
Images belonging to duplicate groups: 42
Extra copies that can be removed: 21
Audit CSV saved.


In [ ]:
import pandas as pd

df = pd.read_csv("/content/plantvillage_md5_audit.csv")

duplicates = df[df["is_duplicate"]].sort_values(
    ["md5", "image_path"]
)

print("Duplicate files:", len(duplicates))
print("Duplicate groups:", duplicates["md5"].nunique())

for md5_hash, group in duplicates.groupby("md5"):
    print("\nMD5:", md5_hash)
    for _, row in group.iterrows():
        print("  Label:", row["label"])
        print("  Path: ", row["image_path"])

In [ ]:
import os
import shutil
import pandas as pd

audit_path = "/content/plantvillage_md5_audit.csv"
source_root = "/content/plantvillage/plantvillage dataset/color"
output_root = "/content/plantvillage_cleaned"

df = pd.read_csv(audit_path)

# Verify that each identical-hash group has only one class label
label_counts = df.groupby("md5")["label"].nunique()
conflicting_hashes = label_counts[label_counts > 1]

if len(conflicting_hashes) > 0:
    raise ValueError(
        f"Cleaning stopped: {len(conflicting_hashes)} hash groups "
        "contain conflicting labels."
    )

# Keep one file per MD5 hash
unique_df = (
    df.sort_values("image_path")
      .drop_duplicates(subset="md5", keep="first")
)

# Copy unique images, preserving class-folder structure
os.makedirs(output_root, exist_ok=True)

for _, row in unique_df.iterrows():
    source = row["image_path"]
    relative_path = os.path.relpath(source, source_root)
    destination = os.path.join(output_root, relative_path)

    os.makedirs(os.path.dirname(destination), exist_ok=True)
    shutil.copy2(source, destination)

print("Original images:", len(df))
print("Cleaned images:", len(unique_df))
print("Redundant copies excluded:", len(df) - len(unique_df))
print("Cleaned dataset path:", output_root)

In [ ]:
import os
import hashlib
from collections import defaultdict

cleaned_root = "/content/plantvillage_cleaned"

hash_to_paths = defaultdict(list)
total_images = 0

for root, _, files in os.walk(cleaned_root):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            path = os.path.join(root, file)

            with open(path, "rb") as f:
                md5 = hashlib.md5(f.read()).hexdigest()

            hash_to_paths[md5].append(path)
            total_images += 1

duplicate_groups = {
    md5: paths
    for md5, paths in hash_to_paths.items()
    if len(paths) > 1
}

print("Images in cleaned dataset:", total_images)
print("Unique MD5 hashes:", len(hash_to_paths))
print("Remaining exact duplicate groups:", len(duplicate_groups))

In [ ]:
!pip install ImageHash

In [ ]:
import os
import pandas as pd
import imagehash
from PIL import Image
from tqdm.auto import tqdm

dataset_root = "/content/plantvillage_cleaned"
output_csv = "/content/plantvillage_phash.csv"

records = []

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_paths = []

for root, _, files in os.walk(dataset_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_paths.append(os.path.join(root, file))

for path in tqdm(image_paths, desc="Computing pHash"):
    try:
        with Image.open(path) as img:
            phash_value = str(imagehash.phash(img.convert("RGB")))

        label = os.path.basename(os.path.dirname(path))

        records.append({
            "image_path": path,
            "label": label,
            "phash": phash_value
        })

    except Exception as e:
        print(f"Could not process {path}: {e}")

phash_df = pd.DataFrame(records)
phash_df.to_csv(output_csv, index=False)

print("Images processed:", len(phash_df))
print("pHash CSV saved to:", output_csv)
print(phash_df.head())

In [ ]:

import pandas as pd
from collections import defaultdict
from itertools import combinations

phash_df = pd.read_csv("/content/plantvillage_phash.csv")

THRESHOLD = 4
output_csv = "/content/plantvillage_phash_candidates.csv"

# Convert each hexadecimal pHash into an integer
hash_to_rows = defaultdict(list)

for idx, row in phash_df.iterrows():
    hash_to_rows[int(row["phash"], 16)].append(idx)

# Build a BK-tree containing unique hashes
tree = None

def hamming(a, b):
    return (a ^ b).bit_count()

def insert(node, value):
    if node is None:
        return [value, {}]

    current, children = node
    distance = hamming(value, current)

    if distance not in children:
        children[distance] = [value, {}]
    else:
        insert(children[distance], value)

    return node

def search(node, value, threshold, found):
    if node is None:
        return

    current, children = node
    distance = hamming(value, current)

    if distance <= threshold:
        found.append((current, distance))

    low = distance - threshold
    high = distance + threshold

    for edge_distance, child in children.items():
        if low <= edge_distance <= high:
            search(child, value, threshold, found)

unique_hashes = list(hash_to_rows.keys())

for value in unique_hashes:
    tree = insert(tree, value)

print("Unique pHashes:", len(unique_hashes))
print("Searching for candidates...")

# Find matching hash pairs
candidate_hash_pairs = {}

for i, value in enumerate(unique_hashes):
    found = []
    search(tree, value, THRESHOLD, found)

    for other, distance in found:
        if value < other:
            candidate_hash_pairs[(value, other)] = distance

# Convert hash pairs into image-pair records
results = []

for (hash_a, hash_b), distance in candidate_hash_pairs.items():
    for idx_a in hash_to_rows[hash_a]:
        for idx_b in hash_to_rows[hash_b]:
            a = phash_df.iloc[idx_a]
            b = phash_df.iloc[idx_b]

            results.append({
                "image_path_1": a["image_path"],
                "label_1": a["label"],
                "image_path_2": b["image_path"],
                "label_2": b["label"],
                "hamming_distance": distance,
                "same_label": a["label"] == b["label"]
            })

# Also include distinct files that have identical pHash values
for value, indices in hash_to_rows.items():
    if len(indices) > 1:
        for idx_a, idx_b in combinations(indices, 2):
            a = phash_df.iloc[idx_a]
            b = phash_df.iloc[idx_b]

            results.append({
                "image_path_1": a["image_path"],
                "label_1": a["label"],
                "image_path_2": b["image_path"],
                "label_2": b["label"],
                "hamming_distance": 0,
                "same_label": a["label"] == b["label"]
            })

result_df = pd.DataFrame(results)

if not result_df.empty:
    result_df = result_df.sort_values(
        "hamming_distance"
    ).drop_duplicates(
        subset=["image_path_1", "image_path_2"]
    )

result_df.to_csv(output_csv, index=False)

print("Candidate pairs found:", len(result_df))
print("Same-label pairs:", int(result_df["same_label"].sum()) if len(result_df) else 0)
print("Different-label pairs:", int((~result_df["same_label"]).sum()) if len(result_df) else 0)
print("Candidate CSV saved to:", output_csv)


In [ ]:

import pandas as pd

input_csv = "/content/plantvillage_phash_candidates.csv"
output_csv = "/content/plantvillage_phash_review.csv"

df = pd.read_csv(input_csv)

# Add columns for manual review
df["review_decision"] = ""
df["review_notes"] = ""

df.to_csv(output_csv, index=False)

print("Total candidate pairs:", len(df))
print("Review CSV saved to:", output_csv)

print("\nColumns:")
print(df.columns.tolist())

print("\nSample:")
print(df.head(5).to_string(index=False))


In [ ]:
import pandas as pd

df = pd.read_csv("/content/plantvillage_phash_review.csv")

print("Candidate pairs by Hamming distance:")
print(df["hamming_distance"].value_counts().sort_index())

print("\nSame-label vs different-label pairs:")
print(df["same_label"].value_counts())

print("\nPairs grouped by distance and label agreement:")
print(
    df.groupby(["hamming_distance", "same_label"])
      .size()
      .unstack(fill_value=0)
)

print("\nPairs with identical perceptual hashes:")
print("Count:", (df["hamming_distance"] == 0).sum())
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
import math

df = pd.read_csv("/content/plantvillage_phash_review.csv")

# Review the closest candidate pairs first
review_df = df[df["hamming_distance"] <= 2].copy()

print("Pairs to inspect:", len(review_df))

# Display 10 pairs per batch
batch_size = 10
batch_number = 0

def show_batch(batch_number):
    start = batch_number * batch_size
    batch = review_df.iloc[start:start + batch_size]

    if batch.empty:
        print("No more pairs in this batch.")
        return

    fig, axes = plt.subplots(
        len(batch), 2,
        figsize=(10, 3 * len(batch))
    )

    if len(batch) == 1:
        axes = [axes]

    for row_axes, (_, pair) in zip(axes, batch.iterrows()):
        for ax, path, label in [
            (row_axes[0], pair["image_path_1"], pair["label_1"]),
            (row_axes[1], pair["image_path_2"], pair["label_2"])
        ]:
            try:
                img = Image.open(path).convert("RGB")
                ax.imshow(img)
                ax.set_title(str(label), fontsize=8)
                ax.axis("off")
            except Exception as e:
                ax.text(0.5, 0.5, f"Could not load image\n{e}",
                        ha="center", va="center")
                ax.axis("off")

        row_axes[0].set_ylabel(
            f"Pair {start + len(row_axes) if False else ''}",
            fontsize=8
        )

        row_axes[0].set_title(
            f'{pair["label_1"]}\nDistance: {pair["hamming_distance"]}',
            fontsize=8
        )
        row_axes[1].set_title(
            f'{pair["label_2"]}\nDistance: {pair["hamming_distance"]}',
            fontsize=8
        )

    plt.tight_layout()
    plt.show()

show_batch(batch_number)
batch_number = 1
show_batch(batch_number)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

review_path = "/content/plantvillage_phash_review.csv"
df = pd.read_csv(review_path)

# Select distance-2 pairs
review_df = df[df["hamming_distance"] == 2].copy().reset_index()

batch_size = 10
batch_number = 0

def show_batch(batch_number):
    start = batch_number * batch_size
    batch = review_df.iloc[start:start + batch_size]

    if batch.empty:
        print("No more pairs in this batch.")
        return

    fig, axes = plt.subplots(len(batch), 2, figsize=(10, 3 * len(batch)))

    for i, (_, pair) in enumerate(batch.iterrows()):
        for ax, path in zip(
            axes[i],
            [pair["image_path_1"], pair["image_path_2"]]
        ):
            try:
                ax.imshow(Image.open(path).convert("RGB"))
                ax.axis("off")
            except Exception as e:
                ax.text(0.5, 0.5, str(e), ha="center")
                ax.axis("off")

        axes[i][0].set_title(
            f'Pair {start+i+1} | {pair["label_1"]}', fontsize=8
        )
        axes[i][1].set_title(
            f'Pair {start+i+1} | {pair["label_2"]}', fontsize=8
        )

    plt.tight_layout()
    plt.show()

show_batch(batch_number)
batch_number = 0
show_batch(batch_number)

In [ ]:

import pandas as pd
import numpy as np
from PIL import Image

# Load all candidate pairs
review_path = "/content/plantvillage_phash_review.csv"
df = pd.read_csv(review_path)

# IMPORTANT: Include ALL pairs, without filtering
candidates = df.copy()

print("Original candidate pairs:", len(df))
print("Pairs selected for analysis:", len(candidates))

# Calculate image resolution and grayscale contrast
def get_metrics(path):
    with Image.open(path) as img:
        width, height = img.size
        gray = np.asarray(img.convert("L"), dtype=np.float32)
        contrast = float(gray.std())

    return width, height, contrast

for suffix in ["1", "2"]:
    metrics = candidates[f"image_path_{suffix}"].apply(get_metrics)

    candidates[
        [f"width_{suffix}", f"height_{suffix}", f"contrast_{suffix}"]
    ] = pd.DataFrame(
        metrics.tolist(), index=candidates.index
    )

# Compare image properties within each pair
candidates["same_resolution"] = (
    (candidates["width_1"] == candidates["width_2"]) &
    (candidates["height_1"] == candidates["height_2"])
)

candidates["contrast_difference"] = abs(
    candidates["contrast_1"] - candidates["contrast_2"]
)

# Rank candidates; do not automatically classify or delete images
candidates = candidates.sort_values(
    by=["hamming_distance", "same_resolution", "contrast_difference"],
    ascending=[True, False, True]
)

# Save a separate CSV; preserve your original review file
output_path = "/content/plantvillage_resolution_contrast_review.csv"
candidates.to_csv(output_path, index=False)

print("\nPairs with matching resolution:",
      int(candidates["same_resolution"].sum()))
print("Total pairs analyzed:", len(candidates))
print("Saved file:", output_path)

display(candidates[
    ["image_path_1", "label_1", "image_path_2", "label_2",
     "hamming_distance", "same_resolution", "contrast_difference"]
].head(15))


In [ ]:

import os
import math
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Input: your existing pHash candidate review CSV
INPUT_CSV = "/content/plantvillage_phash_review.csv"

# Output locations
OUTPUT_DIR = "/content/plantvillage_manual_review"
CONTACT_DIR = os.path.join(OUTPUT_DIR, "contact_sheets")
os.makedirs(CONTACT_DIR, exist_ok=True)

# Read candidate pairs
df = pd.read_csv(INPUT_CSV)

# Sort by Hamming distance: closest perceptual matches first
df["hamming_distance"] = pd.to_numeric(
    df["hamming_distance"], errors="coerce"
)
df = df.sort_values(
    "hamming_distance", ascending=True, kind="stable"
).reset_index(drop=True)

# Assign stable pair IDs for this review sheet
df.insert(
    0, "pair_id",
    [f"P{i:04d}" for i in range(1, len(df) + 1)]
)

# Preserve existing review decisions if present
for col in ["review_decision", "keep_image", "delete_image", "review_notes"]:
    if col not in df.columns:
        df[col] = ""

# Arrange columns for convenient manual review
columns = [
    "pair_id",
    "image_path_1", "label_1",
    "image_path_2", "label_2",
    "hamming_distance",
    "review_decision", "keep_image", "delete_image", "review_notes"
]
df = df[[c for c in columns if c in df.columns]]

# Save review CSV
CSV_PATH = os.path.join(OUTPUT_DIR, "plantvillage_duplicate_review.csv")
df.to_csv(CSV_PATH, index=False)

# Create contact sheets: 10 pairs per sheet
PAIRS_PER_SHEET = 10
THUMBNAIL_SIZE = (180, 180)

def load_image(path):
    # Support paths that may include a leading /content/
    if not os.path.exists(path):
        raise FileNotFoundError(f"Image not found: {path}")
    return Image.open(path).convert("RGB")

for start in range(0, len(df), PAIRS_PER_SHEET):
    batch = df.iloc[start:start + PAIRS_PER_SHEET]

    fig, axes = plt.subplots(
        len(batch), 2,
        figsize=(10, max(3, len(batch) * 3.2)),
        squeeze=False
    )

    for row_idx, (_, row) in enumerate(batch.iterrows()):
        for col_idx, path_col in enumerate(
            ["image_path_1", "image_path_2"]
        ):
            ax = axes[row_idx, col_idx]
            path = row[path_col]

            try:
                img = load_image(str(path))
                img.thumbnail(THUMBNAIL_SIZE)
                ax.imshow(img)
                ax.set_title(
                    f"{'IMAGE A' if col_idx == 0 else 'IMAGE B'}\n"
                    f"Label: {row.get('label_1' if col_idx == 0 else 'label_2', 'Unknown')}",
                    fontsize=9
                )
            except Exception as e:
                ax.text(0.5, 0.5, f"Could not load image\n{e}",
                        ha="center", va="center", wrap=True)

            ax.axis("off")

        fig.text(
            0.5,
            1 - (row_idx + 0.5) / len(batch),
            f"{row['pair_id']} | Hamming distance: "
            f"{row['hamming_distance']}",
            ha="center", va="center", fontsize=10,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray")
        )

    fig.suptitle(
        f"PlantVillage pHash Review — Pairs "
        f"{start + 1} to {start + len(batch)}",
        fontsize=13
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    sheet_path = os.path.join(
        CONTACT_DIR,
        f"contact_sheet_{start + 1:04d}_{start + len(batch):04d}.png"
    )
    fig.savefig(sheet_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

print("Total pairs:", len(df))
print("CSV saved:", CSV_PATH)
print("Contact sheets saved:", CONTACT_DIR)


In [ ]:

import os
import glob
from IPython.display import display, Image

CONTACT_DIR = "/content/plantvillage_manual_review/contact_sheets"

sheets = sorted(glob.glob(os.path.join(CONTACT_DIR, "*.png")))

if not sheets:
    print("No contact sheets found. Check the output folder.")
else:
    # Display the first 10 contact sheets directly in Colab
    for f in sheets[140:150]:
        print(os.path.basename(f))
        display(Image(filename=f))

    print(f"Displayed {min(10, len(sheets))} of {len(sheets)} contact sheets.")


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving plantvillage_duplicate_reviewed.xlsx to plantvillage_duplicate_reviewed.xlsx


In [ ]:

import os
import shutil
import pandas as pd
from collections import defaultdict

# ==========================================
# 1. SETTINGS
# ==========================================

EXCEL_FILE = next(iter(uploaded.keys()))

DATASET_DIR = "/content/plantvillage_cleaned"
OUTPUT_DIR = "/content/plantvillage_deduplicated"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. READ YOUR REVIEWED EXCEL FILE
# ==========================================

df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="plantvillage_duplicate_review"
)

df["review_decision"] = (
    df["review_decision"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Only your confirmed duplicate pairs
duplicates = df[
    df["review_decision"] == "duplicate"
].copy()

print("Confirmed duplicate pairs:", len(duplicates))

# ==========================================
# 3. GROUP OVERLAPPING DUPLICATE PAIRS
# ==========================================

parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a

for _, row in duplicates.iterrows():
    union(
        str(row["image_path_1"]),
        str(row["image_path_2"])
    )

groups = defaultdict(list)

for image in parent:
    groups[find(image)].append(image)

print("Duplicate groups:", len(groups))

# ==========================================
# 4. COLLECT CLASS LABELS
# ==========================================

image_labels = defaultdict(set)

for _, row in df.iterrows():
    image_labels[str(row["image_path_1"])].add(
        str(row["label_1"])
    )
    image_labels[str(row["image_path_2"])].add(
        str(row["label_2"])
    )

# ==========================================
# 5. CREATE KEEP / REMOVE MANIFEST
# ==========================================

manifest = []
remove_images = set()

for group_num, images in enumerate(groups.values(), start=1):
    group_id = f"DUPGROUP_{group_num:04d}"
    images = sorted(images)

    # Default keeper: alphabetically first path
    keeper = images[0]

    for image in images:
        if image == keeper:
            action = "KEEP"
        else:
            action = "REMOVE"
            remove_images.add(image)

        manifest.append({
            "image_path": image,
            "duplicate_group": group_id,
            "action": action,
            "class_labels": "; ".join(
                sorted(image_labels.get(image, set()))
            ),
            "keeper_image_path": keeper
        })

manifest_df = pd.DataFrame(manifest)

# Save reports
manifest_df.to_csv(
    "/content/plantvillage_deduplication_manifest.csv",
    index=False
)

pd.DataFrame({
    "image_path": sorted(remove_images)
}).to_csv(
    "/content/plantvillage_confirmed_removal_list.csv",
    index=False
)

# ==========================================
# 6. FIND ORIGINAL DATASET IMAGES
# ==========================================

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_DIR}. "
        "Update DATASET_DIR to your extracted dataset folder."
    )

all_images = []

for root, _, filenames in os.walk(DATASET_DIR):
    for filename in filenames:
        if filename.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp")
        ):
            all_images.append(
                os.path.join(root, filename)
            )

def normalize_path(path):
    path = str(path).replace("\\", "/")
    marker = "/plantvillage_cleaned/"
    if marker in path:
        return path.split(marker, 1)[1]
    return path

actual_by_relative = {
    os.path.relpath(path, DATASET_DIR).replace("\\", "/"): path
    for path in all_images
}

def resolve_image(excel_path):
    relative = normalize_path(excel_path)

    if relative in actual_by_relative:
        return actual_by_relative[relative]

    matches = [
        actual for rel, actual in actual_by_relative.items()
        if rel.endswith(relative)
    ]

    return matches[0] if len(matches) == 1 else None

remove_actual_paths = set()
unresolved_paths = []

for image in remove_images:
    actual = resolve_image(image)

    if actual:
        remove_actual_paths.add(actual)
    else:
        unresolved_paths.append(image)

# ==========================================
# 7. COPY CLEAN DATASET
# ==========================================

# Original files are NOT deleted.
# Unresolved paths are retained for safety.

copied_count = 0

for source in all_images:
    if source in remove_actual_paths:
        continue

    relative = os.path.relpath(source, DATASET_DIR)
    destination = os.path.join(OUTPUT_DIR, relative)

    os.makedirs(os.path.dirname(destination), exist_ok=True)
    shutil.copy2(source, destination)
    copied_count += 1

# ==========================================
# 8. SAVE ADDITIONAL REPORTS
# ==========================================

pd.DataFrame({
    "unresolved_excel_path": unresolved_paths
}).to_csv(
    "/content/plantvillage_unresolved_paths.csv",
    index=False
)

print("\n===== DEDUPLICATION SUMMARY =====")
print("Images in original dataset:", len(all_images))
print("Confirmed duplicate pairs:", len(duplicates))
print("Duplicate groups:", len(groups))
print("Images marked for removal:", len(remove_images))
print("Removal paths matched:", len(remove_actual_paths))
print("Unresolved paths:", len(unresolved_paths))
print("Images copied to clean dataset:", copied_count)
print("Clean dataset:", OUTPUT_DIR)

# Download reports
files.download(
    "/content/plantvillage_deduplication_manifest.csv"
)
files.download(
    "/content/plantvillage_confirmed_removal_list.csv"
)
files.download(
    "/content/plantvillage_unresolved_paths.csv"
)


Confirmed duplicate pairs: 611
Duplicate groups: 347

===== DEDUPLICATION SUMMARY =====
Images in original dataset: 54284
Confirmed duplicate pairs: 611
Duplicate groups: 347
Images marked for removal: 579
Removal paths matched: 579
Unresolved paths: 0
Images copied to clean dataset: 53705
Clean dataset: /content/plantvillage_deduplicated


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import pandas as pd
from google.colab import files

# ==========================================
# 1. PATHS
# ==========================================

dataset_path = "/content/plantvillage/plantvillage dataset/color"

removal_csv = "/content/plantvillage_confirmed_removal_list.csv"

output_csv = "/content/plantvillage_deduplicated.csv"

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ==========================================
# 2. LOAD REMOVAL LIST
# ==========================================

removal_df = pd.read_csv(removal_csv)

def normalize_key(path):
    path = str(path).strip().replace("\\", "/")
    parts = [p for p in path.split("/") if p]
    if len(parts) < 2:
        return None
    return "/".join(parts[-2:]).casefold()

remove_keys = set(
    removal_df["image_path"]
    .dropna()
    .map(normalize_key)
    .dropna()
)

# ==========================================
# 3. BUILD CSV FROM THE ORIGINAL DATASET
# ==========================================

rows = []
removed_count = 0

for root, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        if not filename.lower().endswith(image_extensions):
            continue

        full_path = os.path.join(root, filename)
        relative_path = os.path.relpath(
            full_path, dataset_path
        ).replace("\\", "/")

        key = normalize_key(full_path)

        # Exclude confirmed duplicates
        if key in remove_keys:
            removed_count += 1
            continue

        rows.append({
            "image_path": full_path,
            "relative_path": relative_path,
            "label": os.path.basename(root),
            "filename": filename
        })

# ==========================================
# 4. SAVE THE COMPLETE DEDUPLICATED CSV
# ==========================================

clean_df = pd.DataFrame(rows)
clean_df = clean_df.sort_values("relative_path").reset_index(drop=True)

clean_df.to_csv(output_csv, index=False)

print("========== RESULTS ==========")
print("Original image count: 54,305 (previous scan)")
print("Images excluded:", removed_count)
print("Images in final CSV:", len(clean_df))
print("Number of classes:", clean_df["label"].nunique())
print("CSV saved to:", output_csv)

# Download the complete CSV
files.download(output_csv)


========== RESULTS ==========
Original image count: 54,305 (previous scan)
Images excluded: 579
Images in final CSV: 53726
Number of classes: 38
CSV saved to: /content/plantvillage_deduplicated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

========== RESULTS ==========
Original image count: 54,305 (previous scan)
Images excluded: 579
Images in final CSV: 53726
Number of classes: 38
CSV saved to: /content/plantvillage_deduplicated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import hashlib
import pandas as pd
from collections import defaultdict
from google.colab import files

DATASET_DIR = "/content/plantvillage/plantvillage dataset/color"
OUTPUT_CSV = "/content/plantvillage_md5_removal_list.csv"

hash_to_paths = defaultdict(list)

for root, _, filenames in os.walk(DATASET_DIR):
    for filename in filenames:
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            path = os.path.join(root, filename)

            md5 = hashlib.md5()
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    md5.update(chunk)

            hash_to_paths[md5.hexdigest()].append(path)

# Retain one image per exact-duplicate group; list only extra copies
remove_paths = []

for paths in hash_to_paths.values():
    if len(paths) > 1:
        remove_paths.extend(sorted(paths)[1:])

pd.DataFrame({"image_path": sorted(remove_paths)}).to_csv(
    OUTPUT_CSV, index=False
)

print("Images scanned:", sum(len(v) for v in hash_to_paths.values()))
print("Unique MD5 hashes:", len(hash_to_paths))
print("Extra exact copies to remove:", len(remove_paths))
print("Saved:", OUTPUT_CSV)




Images scanned: 54305
Unique MD5 hashes: 54284
Extra exact copies to remove: 21
Saved: /content/plantvillage_md5_removal_list.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import pandas as pd
from google.colab import files

# ==========================================
# 1. PATHS
# ==========================================

DATASET_DIR = "/content/plantvillage/plantvillage dataset/color"

NEAR_CSV = "/content/plantvillage_confirmed_removal_list.csv"
MD5_CSV = "/content/plantvillage_md5_removal_list.csv"

OUTPUT_CSV = "/content/plantvillage_final_deduplicated.csv"

EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ==========================================
# 2. NORMALIZE PATHS
# ==========================================

def normalize_key(path):
    path = str(path).strip().replace("\\", "/")
    parts = [p for p in path.split("/") if p]
    if len(parts) < 2:
        return None
    return "/".join(parts[-2:]).casefold()

# ==========================================
# 3. LOAD BOTH REMOVAL LISTS
# ==========================================

near_df = pd.read_csv(NEAR_CSV)
md5_df = pd.read_csv(MD5_CSV)

near_keys = set(
    near_df["image_path"].dropna().map(normalize_key).dropna()
)
md5_keys = set(
    md5_df["image_path"].dropna().map(normalize_key).dropna()
)

# Combine sets so overlapping paths are counted only once
remove_keys = near_keys | md5_keys

print("Near-duplicate paths:", len(near_keys))
print("MD5 removal paths:", len(md5_keys))
print("Overlap between lists:", len(near_keys & md5_keys))
print("Total unique removal paths:", len(remove_keys))

# ==========================================
# 4. BUILD FINAL CSV FROM ORIGINAL DATASET
# ==========================================

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_DIR}")

rows = []
removed_count = 0
matched_removal_keys = set()

for root, _, filenames in os.walk(DATASET_DIR):
    for filename in filenames:
        if not filename.lower().endswith(EXTENSIONS):
            continue

        full_path = os.path.join(root, filename)
        relative_path = os.path.relpath(
            full_path, DATASET_DIR
        ).replace("\\", "/")

        key = normalize_key(full_path)

        if key in remove_keys:
            removed_count += 1
            matched_removal_keys.add(key)
            continue

        rows.append({
            "image_path": full_path,
            "relative_path": relative_path,
            "label": os.path.basename(root),
            "filename": filename
        })

# ==========================================
# 5. SAVE FINAL CSV
# ==========================================

final_df = pd.DataFrame(rows)
final_df = final_df.sort_values("relative_path").reset_index(drop=True)
final_df.to_csv(OUTPUT_CSV, index=False)

unmatched_keys = remove_keys - matched_removal_keys

print("\n========== FINAL RESULTS ==========")
print("Original images scanned:", removed_count + len(final_df))
print("Unique removal paths matched:", len(matched_removal_keys))
print("Images excluded:", removed_count)
print("Unmatched removal paths:", len(unmatched_keys))
print("Final CSV image rows:", len(final_df))
print("Number of classes:", final_df["label"].nunique())
print("Saved to:", OUTPUT_CSV)

# Download only the final CSV
files.download(OUTPUT_CSV)


Near-duplicate paths: 579
MD5 removal paths: 21
Overlap between lists: 0
Total unique removal paths: 600

========== FINAL RESULTS ==========
Original images scanned: 54305
Unique removal paths matched: 600
Images excluded: 600
Unmatched removal paths: 0
Final CSV image rows: 53705
Number of classes: 38
Saved to: /content/plantvillage_final_deduplicated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>